# 00 - Introduction: what we are building

A learning-first walkthrough of a **stateful LangGraph agent** that
ingests Hacker News stories, classifies them into subtopics, embeds
and clusters them, detects trending topics over time, and produces a
human-readable summary.

Every markdown block explains *why* we are doing what the code does.
The goal is that a reader new to LangChain / LangGraph or to agent
orchestration in general can follow the whole pipeline end to end.

## Why Hacker News

Two reasons. First, the tech-community angle: HN's `topstories` and
`newstories` streams are dominated by ML paper releases, tool
announcements, and technical discussion, which is exactly the
category set we want to classify against. Second, the pragmatic
angle: the HN API is a public read-only Firebase JSON endpoint that
needs no authentication, no signup, and no policy approval. It is
documented at <https://github.com/HackerNews/API>.

We considered Reddit first (via r/MachineLearning) but Reddit's 2024
Responsible Builder Policy has made hobbyist API access effectively
unreliable. Hacker News gives us the same problem shape with none of
the auth friction.

## The problem

HN moves fast. Story submissions arrive continuously, mixing paper
releases, product launches, tutorials, opinion pieces, and industry
news. "What is trending this week on the AI/ML slice of HN?" is not
a single-query question: it needs classification, clustering,
temporal comparison, and human-readable summarisation.

The approach here breaks that job into pieces a small LLM (running
locally on Ollama) can handle, orchestrated as a LangGraph state
machine so each step is testable and the "drill down when a topic
spikes" behaviour is a real conditional edge, not a hidden
prompt-engineering trick.

## Framing

**Question.** Given a rolling window of HN stories, produce (a)
per-story subtopic labels, (b) a list of the top-N trending
subtopics vs the previous window, and (c) a short natural-language
summary of what is happening this week.

**Metrics.**

- *Classification accuracy* on a hand-labelled gold set of ~200-300
  stories (notebook 02 builds the gold set; notebooks 03 and 04
  evaluate against it).
- *Cluster purity* against the gold labels for the embedding step
  (notebook 05).
- *Manual verification* of the top-N trending subtopics (notebook
  07) - the trend detector's output is checked by hand.

**Baselines we must beat.**

- Classical TF-IDF + logistic regression (notebook 03) for
  classification.
- A single-prompt LangChain call (no state) for summarisation - we
  should be able to justify the LangGraph state machine over a
  simple prompt chain.

## Roadmap through the notebooks

| # | Notebook | What you will learn |
|---|----------|---------------------|
| 00 | this one | The problem, dataset, framing |
| 01 | `01_fetch_and_eda` | Set up the HN API client, pull stories, describe volume and type distribution |
| 02 | `02_manual_gold_labels` | Hand-label a few hundred stories as ground truth for classification |
| 03 | `03_tfidf_baseline` | Classical TF-IDF + logistic regression classifier, evaluated on the gold set |
| 04 | `04_ollama_llm_classifier` | Few-shot prompt via LangChain against a local Ollama model, same evaluation |
| 05 | `05_embeddings_and_clusters` | Sentence embeddings, KMeans + HDBSCAN, cluster purity vs labels |
| 06 | `06_langgraph_agent` | The full state machine with a conditional drill-down node |
| 07 | `07_trend_report` | Trend detection + generated summary; manual verification of the top-N spikes |

## Status

This notebook is a scaffold. Section-level content lands as we build
each notebook out.

**Next step:** open `01_fetch_and_eda.ipynb`.